In [3]:
!pip install langchain langgraph langchain-openai openai numpy python-dotenv

# Verify installations
import langchain
import langgraph
from langgraph.graph import StateGraph
print(" LangChain installed:", langchain.__version__)
print(" LangGraph installed successfully")
print(" StateGraph imported:", StateGraph)

 LangChain installed: 0.3.27
 LangGraph installed successfully
 StateGraph imported: <class 'langgraph.graph.state.StateGraph'>


In [4]:
import os
from google.colab import userdata

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

print("API key set")

API key set


In [5]:
from typing import TypedDict, List, Optional, Annotated
from dataclasses import dataclass
import operator

@dataclass
class PolicyDocument:
    """
    Represents one policy brief in our corpus.
    """
    doc_id: str
    title: str
    text: str

    def __repr__(self):
        return f"PolicyDocument(id={self.doc_id}, title={self.title[:30]}...)"


@dataclass
class RetrievedDocument:
    """
    A document that was retrieved for a specific query,
    including its relevance score.
    """
    doc: PolicyDocument
    score: float

    def __repr__(self):
        return f"Retrieved(id={self.doc.doc_id}, score={self.score:.3f})"

class AgentState(TypedDict):
    # User input
    question: str

    # Research Agent outputs
    retrieved_docs: List[RetrievedDocument]
    research_summary: Optional[str]
    confidence_score: float

    # Analysis Agent outputs
    analysis: Optional[str]

    # Recommendation Agent outputs
    recommendation: Optional[str]

    # Metadata
    completed: bool
    conversation_history: Annotated[List[str], operator.add]


# Test the data structures
print("Data structures defined")

# Quick test
test_doc = PolicyDocument(
    doc_id="test_001",
    title="Test Policy Brief",
    text="This is a test document about AI policy."
)
print(f"Test document created: {test_doc}")

# Create initial state
initial_state: AgentState = {
    "question": "",
    "retrieved_docs": [],
    "research_summary": None,
    "confidence_score": 0.0,
    "analysis": None,
    "recommendation": None,
    "completed": False,
    "conversation_history": []
}
print(f"Initial state created")
print("State fields:", list(initial_state.keys()))

Data structures defined
Test document created: PolicyDocument(id=test_001, title=Test Policy Brief...)
Initial state created
State fields: ['question', 'retrieved_docs', 'research_summary', 'confidence_score', 'analysis', 'recommendation', 'completed', 'conversation_history']


In [7]:

POLICY_CORPUS = [
    PolicyDocument(
        doc_id="pathology_ai_001",
        title="Deep Learning for Histopathology Image Classification - Validation Study",
        text="""
        A multi-institutional study evaluated deep learning models for automated cancer
        classification from whole slide images (WSIs) in clinical pathology workflows.

        Study Design:
        - Sample: 5,200 whole slide images from 3 academic medical centers
        - Cancer types: Lung adenocarcinoma, colon cancer, breast cancer
        - Validation approach: 5-fold cross-validation with external test set
        - Ground truth: Board-certified pathologists (2-3 per case for consensus)
        - Models tested: CNN architectures (ResNet, EfficientNet, ViT, Swin Transformer)

        Model Performance:
        Ensemble deep learning approach achieved:
        - Overall accuracy: 87.98% across cancer subtypes
        - Cohen's Kappa: 0.8838 (substantial agreement with pathologists)
        - Sensitivity for malignant cases: 92.3%
        - Specificity: 94.1%
        - Processing time: 2-3 minutes per WSI (vs 10-15 minutes human review)

        Clinical Impact:
        - Reduced diagnostic turnaround time by 40% in pilot deployment
        - Flagged 98.7% of high-priority malignant cases for expedited review
        - Provided decision support for 200+ pathologists across institutions
        - Achieved clinical deployment threshold (accuracy > 85%, Kappa > 0.80)

        Subgroup Analysis:
        Model performance varied by:
        - Tissue preparation quality: 91.2% accuracy on optimal prep vs 82.4% on suboptimal
        - Scanner type: 89.1% on Scanner A vs 85.3% on Scanner B (domain shift)
        - Tumor grade: 93.5% for well-differentiated vs 81.2% for poorly-differentiated

        Limitations:
        - Training data primarily from academic centers (limited community hospital representation)
        - Imbalanced dataset (more common cancer types overrepresented)
        - Models struggled with rare histological subtypes (<50 training examples)
        - External validation on only 3 institutions (generalizability unknown)
        - No longitudinal patient outcome data (survival, treatment response)

        Equity and Fairness Considerations:
        - No significant performance differences by patient demographics (age, sex, race)
        - Concern about dataset bias: 78% of samples from majority white populations
        - Recommendation for diverse dataset collection and fairness audits
        - Models should augment, not replace, pathologist expertise

        Deployment Recommendations:
        - Implement as "second reader" system with pathologist final authority
        - Continuous monitoring for model drift and performance degradation
        - Regular retraining with new data to maintain accuracy
        - Clear communication to patients about AI assistance in diagnosis
        """,
    ),

    PolicyDocument(
        doc_id="genomics_ml_002",
        title="Transformer Models for Genomic Sequence Analysis - Splice Site Prediction",
        text="""
        A computational study developed transformer-based models for predicting splice
        sites in genomic sequences, critical for understanding gene expression and disease.

        Study Design:
        - Architecture: BERT-style masked language model adapted for DNA sequences
        - Training data: 4,000-token genomic sequences from human genome (GRCh38)
        - Task: Predict donor and acceptor splice sites with position-level accuracy
        - Comparison models: Traditional CNNs, LSTMs, hybrid CNN-BERT architectures
        - Validation: Hold-out test set plus cross-species validation (mouse, zebrafish)

        Model Architecture:
        Custom "Spliceformer" framework:
        - Tokenization: k-mer encoding (k=3,4,5) and byte-pair encoding (BPE)
        - Positional embeddings: Rotary Position Embeddings (RoPE) and learned Gamma embeddings
        - Context window: 4,096 tokens (approximately 12kb genomic region)
        - Pre-training: Masked language modeling on 10 million genomic sequences
        - Fine-tuning: Supervised learning on annotated splice junction database

        Performance Metrics:
        - Donor site prediction accuracy: 89.3% (Top-1), 96.7% (Top-5)
        - Acceptor site prediction: 86.1% (Top-1), 94.2% (Top-5)
        - Improvement over CNN baseline: +15.2% Top-K accuracy
        - False positive rate: 2.3% (critical for clinical applications)
        - Runtime: 40% faster than previous state-of-art (CUDA optimization)

        Reconstruction Task Performance:
        - Masked token reconstruction: 47.6% accuracy on 4k-token sequences
        - Improved with hybrid CNN-BERT architecture by 8.3 percentage points
        - Model captures long-range genomic dependencies (>1000bp context)

        Clinical Relevance:
        - Splice site mutations implicated in ~15% of genetic diseases
        - Accurate prediction enables:
          * Variant interpretation in clinical genetics
          * Drug target identification
          * Understanding disease mechanisms
          * Personalized medicine applications

        Limitations:
        - Training primarily on well-annotated genes (bias toward studied genes)
        - Performance degrades on non-coding regulatory regions
        - Limited validation on patient-derived pathogenic variants
        - Computational requirements (8 GPUs, 24 hours training time)
        - Interpretability challenges (difficult to explain predictions to clinicians)

        Validation Concerns:
        - Cross-species validation shows 15-20% accuracy drop (species-specific patterns)
        - Model overfits to common splice site motifs (AG/GT), struggles with rare variants
        - Need for larger, more diverse training sets including disease variants

        Ethical Considerations:
        - Models trained on publicly available genomes (consent and privacy concerns)
        - Predictions should not replace experimental validation
        - Risk of over-reliance on computational predictions in clinical settings
        - Need for transparency in model limitations when used in diagnostics
        """,
    ),

    PolicyDocument(
        doc_id="clinical_ai_003",
        title="AI-Powered Clinical Decision Support - Pathology Report Processing",
        text="""
        A healthcare system implemented an AI pipeline for automated extraction and
        structuring of information from semi-structured pathology reports.

        System Overview:
        - Problem: 10,000+ heterogeneous pathology PDFs with inconsistent formatting
        - Goal: Extract diagnosis terms, ICD-10/SNOMED codes, specimen details
        - Approach: Multi-stage pipeline combining OCR, NLP, and LLM-based extraction
        - Deployment: Real-time processing integrated with electronic health record (EHR)

        Technical Pipeline:
        Stage 1: Text Extraction and Cleaning
        - OCR for scanned reports (Tesseract + preprocessing)
        - PDF text extraction for digital reports (pdfplumber, PyPDF2)
        - Cleaning: Remove artifacts, standardize formatting, segment sections
        - Challenge: Handle multiple column layouts, tables, special characters

        Stage 2: Section Segmentation
        - Identify key sections: Diagnosis, Gross Description, Microscopic, Clinical History
        - Regex patterns + heuristic rules
        - Critical: Exclude Notes/CLIA info to avoid false positive diagnoses
        - Validation: Manual review of 500 reports showed 94.2% segmentation accuracy

        Stage 3: Information Extraction
        - Extract structured fields: diagnosis terms, anatomic site, histologic grade
        - Methods tested: Rule-based extraction, BioClinical ModernBERT, GPT-4
        - Best approach: Two-stage correction (UMLS + LLM refinement)
        - Handle negation: "negative for dysplasia" correctly classified as absent finding

        Stage 4: Medical Coding
        - Map diagnosis terms to ICD-10-CM and SNOMED CT codes
        - External API integration: BioPortal, UMLS Metathesaurus
        - Frozen mappings for production reliability (avoid API dependency)
        - Multi-code handling: Some diagnoses map to multiple valid codes

        Performance Results:
        - Field extraction accuracy: 92.2% on 320 TCGA reports (32 cancer types)
        - ICD-10 coding accuracy: 88.7% (exact match), 96.1% (correct code family)
        - SNOMED coding accuracy: 85.3%
        - Processing time: 8 seconds per report vs 5-10 minutes manual coding
        - Estimated time savings: 80% reduction in manual coding burden

        Validation Approach:
        - Gold standard: Manual annotation by clinical coding specialists
        - Test set: Stratified sample across cancer types and report complexity
        - Error analysis: Categorized failure modes (OCR errors, ambiguous terminology, rare codes)
        - Continuous monitoring: Random sample audits (5% of reports) for quality assurance

        Clinical Deployment Considerations:
        - Human-in-the-loop: All extracted codes reviewed by certified coding staff
        - Confidence thresholds: Low-confidence extractions flagged for manual review
        - Audit trail: Complete provenance tracking (source text → extracted field → code)
        - Integration: Bidirectional sync with Epic EHR via HL7/FHIR interfaces

        Error Modes and Mitigation:
        - OCR errors on poor-quality scans (5.2% of cases): Manual fallback
        - Ambiguous terminology: Multiple valid interpretations flagged for human decision
        - Missing codes in reference database: Escalated to medical terminology experts
        - Version drift: Regular updates to ICD-10/SNOMED mappings (annual releases)

        Limitations:
        - Training on TCGA reports (academic pathology) may not generalize to community settings
        - Limited coverage of rare diseases and non-oncology reports
        - Performance dependent on report quality (formatting, completeness)
        - No validation on real-time patient care impact or clinical outcomes

        Privacy and Security:
        - HIPAA-compliant infrastructure (encrypted storage, access controls)
        - De-identification: PHI removed before model training
        - No patient data stored in external APIs
        - Regular security audits and penetration testing

        Equity Implications:
        - Automated coding may reduce disparities from manual coder variability
        - Risk: If model trained on non-representative data, could perpetuate biases
        - Recommendation: Monitor coding accuracy across patient demographics and report types
        """,
    ),
]

print(f"Policy corpus created with {len(POLICY_CORPUS)} documents")
for doc in POLICY_CORPUS:
    print(f"  - {doc.doc_id}: {doc.title}")

Policy corpus created with 3 documents
  - pathology_ai_001: Deep Learning for Histopathology Image Classification - Validation Study
  - genomics_ml_002: Transformer Models for Genomic Sequence Analysis - Splice Site Prediction
  - clinical_ai_003: AI-Powered Clinical Decision Support - Pathology Report Processing


In [9]:
# ========================================
# RETRIEVAL SYSTEM (RAG)
# ========================================

import numpy as np
from openai import OpenAI
import math

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
EMBEDDING_MODEL = "text-embedding-3-small"


def get_embedding(text: str) -> np.ndarray:
    """Convert text to embedding vector."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[text]
    )
    embedding = response.data[0].embedding
    return np.array(embedding, dtype=np.float32)


def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    dot_product = float(np.dot(vec_a, vec_b))
    norm_a = math.sqrt(float(np.dot(vec_a, vec_a)))
    norm_b = math.sqrt(float(np.dot(vec_b, vec_b)))

    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0

    return dot_product / (norm_a * norm_b)


def build_corpus_index(documents: List[PolicyDocument]) -> dict:
    """Build embedding index for document corpus."""
    print("Building corpus index...")

    embeddings = []
    for i, doc in enumerate(documents):
        combined_text = f"{doc.title}\n\n{doc.text}"
        embedding = get_embedding(combined_text)
        embeddings.append(embedding)
        print(f"  Embedded document {i+1}/{len(documents)}: {doc.doc_id}")

    embed_matrix = np.vstack(embeddings)
    print(f"Index built. Matrix shape: {embed_matrix.shape}")

    return {
        'documents': documents,
        'embeddings': embed_matrix
    }


def retrieve_documents(
    question: str,
    corpus_index: dict,
    k: int = 3
) -> List[RetrievedDocument]:
    """Retrieve top-k most relevant documents for a query."""

    # Embed query
    query_embed = get_embedding(question)

    documents = corpus_index['documents']
    doc_embeddings = corpus_index['embeddings']

    # Compute similarities
    results = []
    for i, doc in enumerate(documents):
        doc_embed = doc_embeddings[i]
        score = cosine_similarity(query_embed, doc_embed)
        results.append(RetrievedDocument(doc=doc, score=score))

    # Sort and return top-k
    results.sort(key=lambda x: x.score, reverse=True)
    return results[:k]


# Build index
print("-"*50)
CORPUS_INDEX = build_corpus_index(POLICY_CORPUS)
print(f"Retrieval system ready. Corpus size: {len(CORPUS_INDEX['documents'])}")
print("-"*50)

--------------------------------------------------
Building corpus index...
  Embedded document 1/3: pathology_ai_001
  Embedded document 2/3: genomics_ml_002
  Embedded document 3/3: clinical_ai_003
Index built. Matrix shape: (3, 1536)
Retrieval system ready. Corpus size: 3
--------------------------------------------------


In [10]:
# ========================================
# TEST RETRIEVAL SYSTEM
# ========================================

print("\n" + "-"*50)
print("Testing Retrieval System")
print("-"*50)

# Test queries - update these to match your healthcare corpus
test_queries = [
    "How accurate are deep learning models for cancer diagnosis?",
    "Can transformer models predict splice sites in genomic sequences?",
    "How effective is AI for extracting information from pathology reports?"
]

for i, query in enumerate(test_queries, 1):
    print(f"\nTest {i}: {query}")
    print("-" * 50)

    retrieved = retrieve_documents(query, CORPUS_INDEX, k=3)
    print(f"Retrieved {len(retrieved)} documents:\n")

    for j, result in enumerate(retrieved, 1):
        print(f"{j}. {result.doc.doc_id}")
        print(f"   Title: {result.doc.title[:60]}...")
        print(f"   Score: {result.score:.4f}")

        # Assess relevance
        if result.score > 0.7:
            print(f"   [HIGHLY RELEVANT]")
        elif result.score > 0.5:
            print(f"   [MODERATELY RELEVANT]")
        else:
            print(f"   [LOW RELEVANCE]")
        print()

    # Top match
    top = retrieved[0]
    print(f"Top match: {top.doc.doc_id} (score: {top.score:.4f})")

print("\n" + "-"*50)
print("Retrieval tests complete")
print("-"*50)


--------------------------------------------------
Testing Retrieval System
--------------------------------------------------

Test 1: How accurate are deep learning models for cancer diagnosis?
--------------------------------------------------
Retrieved 3 documents:

1. pathology_ai_001
   Title: Deep Learning for Histopathology Image Classification - Vali...
   Score: 0.6459
   [MODERATELY RELEVANT]

2. clinical_ai_003
   Title: AI-Powered Clinical Decision Support - Pathology Report Proc...
   Score: 0.4807
   [LOW RELEVANCE]

3. genomics_ml_002
   Title: Transformer Models for Genomic Sequence Analysis - Splice Si...
   Score: 0.4341
   [LOW RELEVANCE]

Top match: pathology_ai_001 (score: 0.6459)

Test 2: Can transformer models predict splice sites in genomic sequences?
--------------------------------------------------
Retrieved 3 documents:

1. genomics_ml_002
   Title: Transformer Models for Genomic Sequence Analysis - Splice Si...
   Score: 0.7614
   [HIGHLY RELEVANT]

2. cl

In [11]:
# ========================================
# RESEARCH AGENT
# ========================================

from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
)

print("LLM initialized: gpt-4o-mini")


def research_agent_node(state: AgentState) -> AgentState:
    """
    Research Agent: Retrieves relevant documents and summarizes evidence.
    """
    print("\n--- Research Agent ---")

    question = state["question"]
    print(f"Question: {question}")

    # Retrieve relevant documents
    print("Retrieving documents...")
    retrieved = retrieve_documents(question, CORPUS_INDEX, k=3)

    print(f"Found {len(retrieved)} documents:")
    for doc in retrieved:
        print(f"  - {doc.doc.doc_id} (score: {doc.score:.3f})")

    confidence = retrieved[0].score if retrieved else 0.0
    print(f"Confidence score: {confidence:.3f}")

    # Build context from top documents
    context_parts = []
    for i, ret_doc in enumerate(retrieved[:2], 1):
        if ret_doc.score > 0.5:
            context_parts.append(
                f"DOCUMENT {i} [{ret_doc.doc.doc_id}]:\n"
                f"Title: {ret_doc.doc.title}\n"
                f"Content:\n{ret_doc.doc.text}\n"
            )

    context = "\n\n".join(context_parts)

    # Prompt LLM to summarize
    print("Generating summary...")

    research_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a research assistant trained in empirical science.

Extract and summarize key findings from research documents.

Focus on:
- Study design (RCT? Observational? Sample size?)
- Primary outcomes and effect sizes (with numbers)
- Population studied
- Key findings

Be precise. Use bullet points. Cite specific numbers."""),

        ("user", """User question: {question}

Retrieved research:

{context}

Summarize the key empirical findings relevant to the question.
Focus on concrete results, effect sizes, and study design details.""")
    ])

    chain = research_prompt | llm
    response = chain.invoke({
        "question": question,
        "context": context
    })

    research_summary = response.content

    print(f"Summary generated ({len(research_summary)} chars)")

    # Update state
    state["retrieved_docs"] = retrieved
    state["research_summary"] = research_summary
    state["confidence_score"] = confidence
    state["conversation_history"].append(f"Research Agent: Found {len(retrieved)} relevant documents")

    print("Research Agent complete\n")

    return state


# ========================================
# TEST RESEARCH AGENT
# ========================================

print("\n" + "-"*50)
print("Testing Research Agent")
print("-"*50)

# Update test question to match healthcare corpus
test_state: AgentState = {
    "question": "How accurate are deep learning models for cancer histopathology classification?",
    "retrieved_docs": [],
    "research_summary": None,
    "confidence_score": 0.0,
    "analysis": None,
    "recommendation": None,
    "completed": False,
    "conversation_history": []
}

print(f"Test question: {test_state['question']}\n")

result_state = research_agent_node(test_state)

# Check results
print("-"*50)
print("Results")
print("-"*50)
print(f"Retrieved docs: {len(result_state['retrieved_docs'])}")
print(f"Summary length: {len(result_state['research_summary'])} chars")
print(f"Confidence: {result_state['confidence_score']:.3f}")
print(f"\nFull Summary:")
print("-" * 50)
print(result_state['research_summary'])
print("-" * 50)

LLM initialized: gpt-4o-mini

--------------------------------------------------
Testing Research Agent
--------------------------------------------------
Test question: How accurate are deep learning models for cancer histopathology classification?


--- Research Agent ---
Question: How accurate are deep learning models for cancer histopathology classification?
Retrieving documents...
Found 3 documents:
  - pathology_ai_001 (score: 0.691)
  - clinical_ai_003 (score: 0.461)
  - genomics_ml_002 (score: 0.377)
Confidence score: 0.691
Generating summary...
Summary generated (2185 chars)
Research Agent complete

--------------------------------------------------
Results
--------------------------------------------------
Retrieved docs: 3
Summary length: 2185 chars
Confidence: 0.691

Full Summary:
--------------------------------------------------
### Key Findings from Document 1: Deep Learning for Histopathology Image Classification - Validation Study

- **Study Design:**
  - Type: Multi-i

In [12]:
# ========================================
# ANALYSIS AGENT
# ========================================

def analysis_agent_node(state: AgentState) -> AgentState:
    """
    Analysis Agent: Critically evaluates study design, validity, and limitations.
    """
    print("\n--- Analysis Agent ---")

    question = state["question"]
    research_summary = state["research_summary"]
    confidence_score = state["confidence_score"]
    retrieved_docs = state["retrieved_docs"]

    print(f"Question: {question}")
    print(f"Confidence: {confidence_score:.3f}")
    print(f"Analyzing evidence from {len(retrieved_docs)} documents")

    # Build context
    top_doc = retrieved_docs[0] if retrieved_docs else None

    context = f"""
RESEARCH SUMMARY:
{research_summary}

CONFIDENCE SCORE: {confidence_score:.3f}
"""

    if top_doc:
        context += f"""
TOP SOURCE: {top_doc.doc.doc_id} (score: {top_doc.score:.3f})
"""

    print("Performing critical analysis...")

    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a research methodologist.

Critically analyze empirical evidence across these dimensions:

1. Study Design Quality
   - RCT, quasi-experimental, or observational?
   - Strengths and weaknesses

2. Causal Claims
   - Can we claim causation or only correlation?
   - Strength of causal evidence
   - Confounding factors

3. Internal Validity
   - Selection bias, measurement issues, attrition

4. External Validity
   - Population studied
   - Generalizability and boundary conditions

5. Equity and Fairness
   - Effects across demographic groups
   - Disparate impacts

6. Gaps and Limitations
   - Missing evidence
   - Follow-up studies needed

Be specific. Cite details. Be balanced."""),

        ("user", """User question: {question}

{context}

Provide a structured critical analysis focusing on:
- How much we can trust these findings
- What the findings do and don't tell us
- What additional evidence would strengthen the case""")
    ])

    chain = analysis_prompt | llm
    response = chain.invoke({
        "question": question,
        "context": context
    })

    analysis = response.content
    print(f"Analysis generated ({len(analysis)} chars)")

    # Update state
    state["analysis"] = analysis
    state["conversation_history"].append("Analysis Agent: Completed critical evaluation")

    print("Analysis Agent complete\n")

    return state


# ========================================
# TEST ANALYSIS AGENT
# ========================================

print("\n" + "-"*50)
print("Testing Analysis Agent")
print("-"*50)

print(f"Analyzing: {result_state['question']}\n")

analyzed_state = analysis_agent_node(result_state)

print("\n" + "-"*50)
print("Results")
print("-"*50)
print(f"Analysis length: {len(analyzed_state['analysis'])} chars")
print(f"\nFull Analysis:")
print("-" * 50)
print(analyzed_state['analysis'])
print("-" * 50)


--------------------------------------------------
Testing Analysis Agent
--------------------------------------------------
Analyzing: How accurate are deep learning models for cancer histopathology classification?


--- Analysis Agent ---
Question: How accurate are deep learning models for cancer histopathology classification?
Confidence: 0.691
Analyzing evidence from 3 documents
Performing critical analysis...
Analysis generated (4707 chars)
Analysis Agent complete


--------------------------------------------------
Results
--------------------------------------------------
Analysis length: 4707 chars

Full Analysis:
--------------------------------------------------
### Critical Analysis of Deep Learning Models for Cancer Histopathology Classification

#### 1. Study Design Quality
- **Type:** The study employed a multi-institutional validation design, which is a strength as it enhances the generalizability of findings across different clinical settings. The use of 5-fold cross-va

In [13]:
# ========================================
# RECOMMENDATION AGENT
# ========================================

def recommendation_agent_node(state: AgentState) -> AgentState:
    """
    Recommendation Agent: Synthesizes evidence into actionable recommendations.
    """
    print("\n--- Recommendation Agent ---")

    question = state["question"]
    research_summary = state["research_summary"]
    analysis = state["analysis"]
    confidence_score = state["confidence_score"]

    print(f"Question: {question}")
    print(f"Confidence: {confidence_score:.3f}")
    print("Synthesizing recommendations...")

    # Build context
    context = f"""
ORIGINAL QUESTION:
{question}

EMPIRICAL EVIDENCE:
{research_summary}

CRITICAL ANALYSIS:
{analysis}

CONFIDENCE: {confidence_score:.3f}
"""

    print("Generating recommendations...")

    recommendation_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a senior policy advisor.

Translate empirical research into actionable recommendations.

For each recommendation, provide:

1. Recommendation Statement
   - Clear, specific action

2. Target Population
   - Who this applies to
   - Geographic scope

3. Implementation Approach
   - Rollout strategy (pilot/phased/universal)
   - Key partners and stakeholders
   - Timeline

4. Evaluation Plan
   - Success metrics
   - Experimental design (RCT if appropriate)
   - Primary and secondary outcomes
   - Sample size and duration

5. Guardrails and Risk Mitigation
   - Potential problems
   - Prevention strategies
   - Monitoring and oversight
   - Fairness checks

6. Limitations Acknowledged
   - Caveats for decision-makers
   - Key assumptions

Provide 2-3 concrete recommendations. Be specific and operational."""),

        ("user", """{context}

Provide actionable recommendations based on this evidence and analysis.
Be specific about implementation and evaluation.
Acknowledge limitations.""")
    ])

    chain = recommendation_prompt | llm
    response = chain.invoke({"context": context})

    recommendation = response.content
    print(f"Recommendations generated ({len(recommendation)} chars)")

    # Update state
    state["recommendation"] = recommendation
    state["completed"] = True
    state["conversation_history"].append("Recommendation Agent: Generated recommendations")

    print("Recommendation Agent complete")
    print("Workflow completed\n")

    return state


# ========================================
# TEST RECOMMENDATION AGENT
# ========================================

print("\n" + "-"*50)
print("Testing Recommendation Agent")
print("-"*50)

print(f"Creating recommendations for: {analyzed_state['question']}\n")

final_state = recommendation_agent_node(analyzed_state)

print("\n" + "-"*50)
print("Results")
print("-"*50)
print(f"Recommendations length: {len(final_state['recommendation'])} chars")
print(f"Workflow completed: {final_state['completed']}")
print(f"\nFull Recommendations:")
print("-" * 50)
print(final_state['recommendation'])
print("-" * 50)

print("\nConversation History:")
for msg in final_state['conversation_history']:
    print(f"  - {msg}")


--------------------------------------------------
Testing Recommendation Agent
--------------------------------------------------
Creating recommendations for: How accurate are deep learning models for cancer histopathology classification?


--- Recommendation Agent ---
Question: How accurate are deep learning models for cancer histopathology classification?
Confidence: 0.691
Synthesizing recommendations...
Generating recommendations...
Recommendations generated (7077 chars)
Recommendation Agent complete
Workflow completed


--------------------------------------------------
Results
--------------------------------------------------
Recommendations length: 7077 chars
Workflow completed: True

Full Recommendations:
--------------------------------------------------
### Recommendation 1: Implement Deep Learning Models as a Second Reader System in Community Hospitals

1. **Recommendation Statement:**
   - Deploy deep learning models for cancer histopathology classification as a "second 

In [14]:
# ========================================
# LANGGRAPH ORCHESTRATOR
# ========================================

from langgraph.graph import StateGraph, END
from typing import Literal

print("\n" + "-"*50)
print("Building LangGraph Orchestrator")
print("-"*50)


def route_after_research(state: AgentState) -> Literal["analysis", "end"]:
    """
    Route based on confidence score.
    - confidence >= 0.5: proceed to analysis
    - confidence < 0.5: insufficient evidence, end
    """
    confidence = state.get("confidence_score", 0.0)

    if confidence >= 0.5:
        print(f"Confidence {confidence:.3f} >= 0.5 - Proceeding to Analysis")
        return "analysis"
    else:
        print(f"Confidence {confidence:.3f} < 0.5 - Insufficient evidence")
        state["recommendation"] = f"Insufficient evidence (confidence: {confidence:.3f}). Please rephrase or try a different question."
        state["completed"] = True
        return "end"


# Build workflow
workflow = StateGraph(AgentState)

print("Adding nodes...")
workflow.add_node("research", research_agent_node)
workflow.add_node("analysis", analysis_agent_node)
workflow.add_node("recommendation", recommendation_agent_node)
print("  - research")
print("  - analysis")
print("  - recommendation")

print("\nAdding edges...")
workflow.set_entry_point("research")
print("  - Entry: research")

workflow.add_conditional_edges(
    "research",
    route_after_research,
    {
        "analysis": "analysis",
        "end": END
    }
)
print("  - research -> analysis (if confidence >= 0.5)")

workflow.add_edge("analysis", "recommendation")
print("  - analysis -> recommendation")

workflow.add_edge("recommendation", END)
print("  - recommendation -> END")

print("\nCompiling workflow...")
app = workflow.compile()
print("Workflow compiled successfully")
print("-"*50)


--------------------------------------------------
Building LangGraph Orchestrator
--------------------------------------------------
Adding nodes...
  - research
  - analysis
  - recommendation

Adding edges...
  - Entry: research
  - research -> analysis (if confidence >= 0.5)
  - analysis -> recommendation
  - recommendation -> END

Compiling workflow...
Workflow compiled successfully
--------------------------------------------------


In [16]:
# ========================================
# TEST COMPLETE WORKFLOW
# ========================================

print("\n" + "-"*50)
print("Testing Complete Workflow")
print("-"*50)

# Update test question to match healthcare corpus
test_question = "How can deep learning improve cancer diagnosis accuracy in clinical pathology?"

initial_state: AgentState = {
    "question": test_question,
    "retrieved_docs": [],
    "research_summary": None,
    "confidence_score": 0.0,
    "analysis": None,
    "recommendation": None,
    "completed": False,
    "conversation_history": []
}

print(f"\nQuestion: {test_question}")
print("Running workflow...\n")
print("-"*50)

# Run workflow
final_result = app.invoke(initial_state)

print("\n" + "-"*50)
print("Workflow Complete")
print("-"*50)

print(f"\nCompleted: {final_result['completed']}")
print(f"Confidence: {final_result['confidence_score']:.3f}")
print(f"Documents used: {len(final_result['retrieved_docs'])}")

print("\n" + "-"*50)
print("Final Output")
print("-"*50)

print("\nEvidence Summary:")
print("-" * 50)
print(final_result['research_summary'])

print("\nCritical Analysis:")
print("-" * 50)
print(final_result['analysis'])

print("\nRecommendations:")
print("-" * 50)
print(final_result['recommendation'])

print("\n" + "-"*50)


--------------------------------------------------
Testing Complete Workflow
--------------------------------------------------

Question: How can deep learning improve cancer diagnosis accuracy in clinical pathology?
Running workflow...

--------------------------------------------------

--- Research Agent ---
Question: How can deep learning improve cancer diagnosis accuracy in clinical pathology?
Retrieving documents...
Found 3 documents:
  - pathology_ai_001 (score: 0.730)
  - clinical_ai_003 (score: 0.570)
  - genomics_ml_002 (score: 0.354)
Confidence score: 0.730
Generating summary...
Summary generated (3200 chars)
Research Agent complete

Confidence 0.730 >= 0.5 - Proceeding to Analysis

--- Analysis Agent ---
Question: How can deep learning improve cancer diagnosis accuracy in clinical pathology?
Confidence: 0.730
Analyzing evidence from 3 documents
Performing critical analysis...
Analysis generated (4590 chars)
Analysis Agent complete


--- Recommendation Agent ---
Question: 

In [17]:
# ========================================
# CHAT INTERFACE
# ========================================

def ask_question(question: str) -> dict:
    """
    Simple interface to the multi-agent system.
    Returns structured results.
    """
    state: AgentState = {
        "question": question,
        "retrieved_docs": [],
        "research_summary": None,
        "confidence_score": 0.0,
        "analysis": None,
        "recommendation": None,
        "completed": False,
        "conversation_history": []
    }

    result = app.invoke(state)

    return {
        "question": question,
        "confidence": result["confidence_score"],
        "evidence": result["research_summary"],
        "analysis": result["analysis"],
        "recommendations": result["recommendation"],
        "sources": [
            {
                "doc_id": doc.doc.doc_id,
                "title": doc.doc.title,
                "score": doc.score
            }
            for doc in result["retrieved_docs"]
        ]
    }


def interactive_chat():
    """Interactive chat loop."""
    print("\n" + "-"*50)
    print("Healthcare AI Research Assistant")
    print("-"*50)
    print("\nAsk questions about AI in healthcare.")
    print("Type 'quit' to exit.\n")

    while True:
        question = input("Your question: ").strip()

        if question.lower() in ['quit', 'exit', 'q']:
            print("\nGoodbye!")
            break

        if not question:
            continue

        print("\nProcessing...\n")

        try:
            result = ask_question(question)

            print("-"*50)
            print("Results")
            print("-"*50)

            print(f"\nConfidence: {result['confidence']:.3f}")
            print(f"Sources:")
            for src in result['sources'][:2]:
                print(f"  - {src['title'][:60]}... (score: {src['score']:.3f})")

            print("\nEvidence:")
            print("-"*50)
            evidence = result['evidence']
            print(evidence[:500] + "..." if len(evidence) > 500 else evidence)

            print("\nRecommendations:")
            print("-"*50)
            recs = result['recommendations']
            print(recs[:800] + "..." if len(recs) > 800 else recs)

            print("\n" + "-"*50 + "\n")

        except Exception as e:
            print(f"\nError: {e}\n")


# Quick test
print("\n" + "-"*50)
print("Testing Interface")
print("-"*50)

# Update test question to healthcare
test_result = ask_question("How accurate are AI models for extracting pathology report information?")

print(f"\nQuestion processed")
print(f"Confidence: {test_result['confidence']:.3f}")
print(f"Sources: {len(test_result['sources'])}")
print(f"Evidence length: {len(test_result['evidence'])} chars")
print(f"Recommendations length: {len(test_result['recommendations'])} chars")

print(f"\nTop source: {test_result['sources'][0]['title']}")
print(f"Score: {test_result['sources'][0]['score']:.3f}")
print("-"*50)


--------------------------------------------------
Testing Interface
--------------------------------------------------

--- Research Agent ---
Question: How accurate are AI models for extracting pathology report information?
Retrieving documents...
Found 3 documents:
  - clinical_ai_003 (score: 0.643)
  - pathology_ai_001 (score: 0.587)
  - genomics_ml_002 (score: 0.436)
Confidence score: 0.643
Generating summary...
Summary generated (2912 chars)
Research Agent complete

Confidence 0.643 >= 0.5 - Proceeding to Analysis

--- Analysis Agent ---
Question: How accurate are AI models for extracting pathology report information?
Confidence: 0.643
Analyzing evidence from 3 documents
Performing critical analysis...
Analysis generated (4284 chars)
Analysis Agent complete


--- Recommendation Agent ---
Question: How accurate are AI models for extracting pathology report information?
Confidence: 0.643
Synthesizing recommendations...
Generating recommendations...
Recommendations generated (6901 

In [19]:
# ========================================
# VALIDATION & EVALUATION
# ========================================

print("\nValidation")
print("-"*50)

# Test cases
VALIDATION_SET = [
    {
        "question": "How accurate are deep learning models for cancer diagnosis?",
        "expected_doc": "pathology_ai_001",
        "key_facts": ["87.98%", "Cohen's Kappa", "whole slide", "ensemble"]
    },
    {
        "question": "Can transformer models predict splice sites accurately?",
        "expected_doc": "genomics_ml_002",
        "key_facts": ["Spliceformer", "donor site", "acceptor site", "89.3%"]
    },
    {
        "question": "How effective is AI for pathology report extraction?",
        "expected_doc": "clinical_ai_003",
        "key_facts": ["92.2%", "ICD-10", "SNOMED", "field extraction"]
    }
]


def test_retrieval():
    """Check if retrieval gets the right documents."""
    print("\nRetrieval test:")
    correct = 0

    for i, test in enumerate(VALIDATION_SET, 1):
        retrieved = retrieve_documents(test["question"], CORPUS_INDEX, k=1)
        top_doc = retrieved[0].doc.doc_id if retrieved else None
        is_correct = top_doc == test["expected_doc"]
        correct += is_correct

        print(f"  {i}. {test['expected_doc']}: {'pass' if is_correct else 'fail'}")

    print(f"Accuracy: {correct}/{len(VALIDATION_SET)}")
    return correct / len(VALIDATION_SET)


def check_facts():
    """See if key facts show up in summaries."""
    print("\nFact checking:")
    total = 0
    found = 0

    for test in VALIDATION_SET:
        state: AgentState = {
            "question": test["question"],
            "retrieved_docs": [],
            "research_summary": None,
            "confidence_score": 0.0,
            "analysis": None,
            "recommendation": None,
            "completed": False,
            "conversation_history": []
        }

        result = research_agent_node(state)
        summary = result["research_summary"].lower()

        for fact in test["key_facts"]:
            total += 1
            if fact.lower() in summary:
                found += 1

    print(f"Facts found: {found}/{total}")
    return found / total


def compare_approaches():
    """Compare with vs without retrieval."""
    print("\nComparing approaches:")

    q = "How accurate are deep learning models for histopathology?"

    # With retrieval
    state: AgentState = {
        "question": q,
        "retrieved_docs": [],
        "research_summary": None,
        "confidence_score": 0.0,
        "analysis": None,
        "recommendation": None,
        "completed": False,
        "conversation_history": []
    }
    result = research_agent_node(state)
    summary_with = result["research_summary"]

    # Without retrieval - just ask LLM
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a research assistant."),
        ("user", "{question}\n\nSummarize relevant evidence.")
    ])
    chain = prompt | llm
    summary_without = chain.invoke({"question": q}).content

    # Check for specific facts
    facts = ["87.98%", "Cohen's Kappa", "whole slide", "ensemble", "5,200", "academic centers"]

    with_count = sum(1 for f in facts if f.lower() in summary_with.lower())
    without_count = sum(1 for f in facts if f.lower() in summary_without.lower())

    print(f"With retrieval: {with_count}/{len(facts)} facts")
    print(f"Without retrieval: {without_count}/{len(facts)} facts")
    print(f"Improvement: {((with_count - without_count) / len(facts) * 100):.0f}pp")

    return with_count / len(facts) - without_count / len(facts)


# Run tests
print("-"*50)
retrieval_acc = test_retrieval()
fact_rate = check_facts()
improvement = compare_approaches()

print("\nSummary:")
print(f"- Retrieval: {retrieval_acc*100:.0f}%")
print(f"- Fact accuracy: {fact_rate*100:.0f}%")
print(f"- Improvement: +{improvement*100:.0f}pp")
print("-"*50)


Validation
--------------------------------------------------
--------------------------------------------------

Retrieval test:
  1. pathology_ai_001: pass
  2. genomics_ml_002: pass
  3. clinical_ai_003: pass
Accuracy: 3/3

Fact checking:

--- Research Agent ---
Question: How accurate are deep learning models for cancer diagnosis?
Retrieving documents...
Found 3 documents:
  - pathology_ai_001 (score: 0.646)
  - clinical_ai_003 (score: 0.481)
  - genomics_ml_002 (score: 0.434)
Confidence score: 0.646
Generating summary...
Summary generated (2052 chars)
Research Agent complete


--- Research Agent ---
Question: Can transformer models predict splice sites accurately?
Retrieving documents...
Found 3 documents:
  - genomics_ml_002 (score: 0.727)
  - clinical_ai_003 (score: 0.303)
  - pathology_ai_001 (score: 0.244)
Confidence score: 0.727
Generating summary...
Summary generated (1704 chars)
Research Agent complete


--- Research Agent ---
Question: How effective is AI for pathology rep